# Part 2 — Step 6: Patch Extraction

Creates a binary classification dataset from ACNE04 bounding boxes:

- **Positive (acne)**: crop each bounding box region, resize to 224×224
- **Negative (no_acne)**: randomly crop same-sized regions with zero overlap with any GT box

Output (standard PyTorch ImageFolder format):
```
data/patches/
├── train/
│   ├── acne/
│   └── no_acne/
└── val/
    ├── acne/
    └── no_acne/
```

**Run first:** `roboflow_loader.py --download` to populate `data/acne04/`

In [ ]:
import os
from pathlib import Path

if Path('/content').exists():
    os.chdir('/content/AcneDetection')
else:
    if Path('notebooks').exists():
        pass  # already at repo root
print(f'Working directory: {os.getcwd()}')

In [ ]:
import json
import random
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

%matplotlib inline

DATA_DIR   = Path("data/acne04")
PATCH_DIR  = Path("data/patches")
PATCH_SIZE = 224   # EfficientNet-B0 input size
MIN_BOX    = 20    # skip bboxes smaller than this in either dimension
NEG_SIZE   = 90    # crop size for negatives (≈ median bbox size in ACNE04)
SEED       = 42

SPLITS = {"train": "train", "valid": "val"}

random.seed(SEED)

In [ ]:
def has_overlap(box, gt_boxes):
    for g in gt_boxes:
        if box[0] < g[2] and box[2] > g[0] and box[1] < g[3] and box[3] > g[1]:
            return True
    return False

def sample_negative(img_w, img_h, gt_boxes, size, max_tries=100):
    for _ in range(max_tries):
        x1 = random.randint(0, max(0, img_w - size))
        y1 = random.randint(0, max(0, img_h - size))
        candidate = [x1, y1, x1 + size, y1 + size]
        if not has_overlap(candidate, gt_boxes):
            return candidate
    return None

def extract_patches(acne04_split, patch_split):
    pos_dir = PATCH_DIR / patch_split / "acne"
    neg_dir = PATCH_DIR / patch_split / "no_acne"
    pos_dir.mkdir(parents=True, exist_ok=True)
    neg_dir.mkdir(parents=True, exist_ok=True)

    with open(DATA_DIR / acne04_split / "_annotations.coco.json") as f:
        coco = json.load(f)

    ann_map = {}
    for ann in coco["annotations"]:
        ann_map.setdefault(ann["image_id"], []).append(ann)

    pos_count = neg_count = skipped = 0

    for meta in coco["images"]:
        img_id = meta["id"]
        anns   = ann_map.get(img_id, [])
        if not anns:
            continue

        img      = Image.open(DATA_DIR / acne04_split / meta["file_name"]).convert("RGB")
        img_w, img_h = img.size
        gt_boxes = []

        for ann in anns:
            x, y, w, h = ann["bbox"]
            if w < MIN_BOX or h < MIN_BOX:
                skipped += 1
                continue
            x1, y1 = int(x), int(y)
            x2, y2 = min(int(x + w), img_w), min(int(y + h), img_h)
            gt_boxes.append([x1, y1, x2, y2])
            patch = img.crop((x1, y1, x2, y2)).resize((PATCH_SIZE, PATCH_SIZE), Image.BILINEAR)
            patch.save(pos_dir / f"{img_id}_{ann['id']}.jpg", quality=90)
            pos_count += 1

        for i in range(len(gt_boxes)):
            box = sample_negative(img_w, img_h, gt_boxes, NEG_SIZE)
            if box is None:
                continue
            x1, y1, x2, y2 = box
            patch = img.crop((x1, y1, x2, y2)).resize((PATCH_SIZE, PATCH_SIZE), Image.BILINEAR)
            patch.save(neg_dir / f"{img_id}_neg{i}.jpg", quality=90)
            neg_count += 1

    print(f"[{acne04_split}]  acne={pos_count}  no_acne={neg_count}  skipped(tiny)={skipped}")

print("Extracting patches...")
for acne04_split, patch_split in SPLITS.items():
    extract_patches(acne04_split, patch_split)
print("Done.")

In [ ]:
# Verify counts
for split in ["train", "val"]:
    for cls in ["acne", "no_acne"]:
        files = list((PATCH_DIR / split / cls).glob("*.jpg"))
        print(f"  data/patches/{split}/{cls}: {len(files)} images")

# Show sample patches
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for row, cls in enumerate(["acne", "no_acne"]):
    files = random.sample(list((PATCH_DIR / "train" / cls).glob("*.jpg")), 4)
    for col, f in enumerate(files):
        axes[row][col].imshow(Image.open(f))
        axes[row][col].set_title(cls, fontsize=9)
        axes[row][col].axis("off")
plt.suptitle("Sample patches — train set", fontsize=13)
plt.tight_layout()
plt.show()